# Supervised Binary Classification & Distance Regression on CSI Windows **with AveCSI + Phase Localization**


Train a 1D CNN on MATLAB-exported windows (`X_window_*.npy`) with channel-wise AveCSI normalization,
use raw phase to estimate distance to anchor A, and jointly fine-tune classification + localization heads.


## 1) Setup & configuration

In [ ]:
import os, json
from glob import glob
from collections import Counter
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers, models, regularizers
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

tf.keras.backend.clear_session()
for g in tf.config.experimental.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass

print("TensorFlow:", tf.__version__)

BASE        = "/home/tonyliao/WIFI_SENSING_LOCATION/PreCNN_Dataset/"
TRAIN_ROOT  = os.path.join(BASE, "training_set")
VAL_ROOT    = os.path.join(BASE, "val_set")
TEST_ROOT   = os.path.join(BASE, "test_set")

CLASS_NAMES = ["Empty", "LocationA"]
CLASS_FOLDERS = None
NUM_CLASSES = len(CLASS_NAMES)

FORCE_LAYOUT = "TC"   # MATLAB exports [time, channels]
CROP_LEN     = None   # If None pad to max T; else force to this T

FEATURE_MODE = "amp+phase+sin_cos"  # options: 'amp', 'amp+phase', 'amp+sin_cos', 'amp+phase+sin_cos'
VALID_FEATURE_MODES = {"amp", "amp+phase", "amp+sin_cos", "amp+phase+sin_cos"}
if FEATURE_MODE not in VALID_FEATURE_MODES:
    raise ValueError(f"Unsupported FEATURE_MODE={FEATURE_MODE}. Valid: {sorted(VALID_FEATURE_MODES)}")

# Anchor-distance configuration

ANCHOR_REFERENCE_ROOT  = None  # Optional path to source anchor windows
ANCHOR_REFERENCE_EMPTY_CLASS = "Empty"
ANCHOR_CLASS_FOLDERS   = None
ENABLE_ANCHOR_DISTANCE = True   # Toggle the auxiliary regression head
ANCHOR_LOCATION_NAME   = "A"    # Human-readable tag for reports
WAVELENGTH_METERS      = 0.032  # Approximate 5.8 GHz WiFi wavelength (meters)
DISTANCE_LOSS_WEIGHT   = 0.2    # Relative weight for distance regression during training
STANDARDIZE_DISTANCE   = True   # Z-score the distance target using the training set statistics

AVECSI_PATH = os.path.join(BASE, "avecsi_empty.npz")
MATLAB_AVE_NAME = "avecsi_M.npy"  # optional fallback

BATCH_SIZE   = 128
EPOCHS       = 40
LR           = 5e-4
WEIGHT_DECAY = 1e-4

OUT_DIR = "/home/tonyliao/WIFI_SENSING_LOCATION/runs_windows_supervised_avecsi"
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Feature mode: {FEATURE_MODE}")
print(f"Anchor distance head: {ENABLE_ANCHOR_DISTANCE}")


TensorFlow: 2.19.0


## 2) Utilities

In [29]:

def list_window_files(root_dir: str, class_names, class_dir_map=None):
    files, labels = [], []
    if not root_dir or not os.path.isdir(root_dir):
        return files, np.array([], dtype=np.int32)
    for ci, cname in enumerate(class_names):
        subdir = class_dir_map.get(cname, cname) if class_dir_map else cname
        patt = os.path.join(root_dir, subdir, "**", "X_window_*")
        fns = [f for f in glob(patt, recursive=True) if os.path.isfile(f)]
        if len(fns) == 0:
            warn_root = os.path.join(root_dir, subdir)
            print(f"[WARN] no windows for class '{cname}' under {warn_root}")
        files.extend(sorted(fns))
        labels.extend([ci] * len(fns))
    return files, np.array(labels, dtype=np.int32)


def get_display_class_names(classes):
    names = list(classes)
    if names:
        first = names[0]
        if isinstance(first, str) and first.lower() == "empty":
            names[0] = "Empty"
    if len(names) > 1:
        second = names[1]
        if isinstance(second, str) and second.lower() == "stationary":
            names[1] = "Stationary (Have people inside)"
    return names

def ensure_channels_first(arr, force_layout="TC"):
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D array, got shape {arr.shape}")
    if force_layout == "TC":
        return arr.T
    if force_layout == "CT":
        return arr
    raise ValueError(f"Unknown force_layout '{force_layout}'")


def related_window_path(path, stem):
    base_dir, fname = os.path.split(path)
    if not fname.startswith("X_window_"):
        raise ValueError(f"Unexpected window filename: {fname}")
    suffix = fname[len("X_window_"):]
    return os.path.join(base_dir, f"{stem}{suffix}")


def wants_raw_phase(mode: str) -> bool:
    return "phase" in mode and mode != "amp+sin_cos"


def wants_sin_cos(mode: str) -> bool:
    return "sin_cos" in mode


def load_window_components(path, feature_mode="amp", force_layout="TC"):
    amp = ensure_channels_first(np.load(path), force_layout=force_layout)
    extras = []
    names = []
    missing = set()

    if feature_mode != "amp":
        if wants_raw_phase(feature_mode):
            phase_path = related_window_path(path, "Xphase_window_")
            if os.path.isfile(phase_path):
                phase = ensure_channels_first(np.load(phase_path), force_layout=force_layout)
                extras.append(phase)
                names.append("phase")
            else:
                missing.add("phase")
                extras.append(np.zeros_like(amp, dtype=np.float32))
                names.append("phase")

        if wants_sin_cos(feature_mode):
            sc_path = related_window_path(path, "XphaseSC_window_")
            if os.path.isfile(sc_path):
                sc = np.load(sc_path)
                if sc.ndim != 3 or sc.shape[-1] != 2:
                    raise ValueError(f"{sc_path} expected shape [T, S, 2], got {sc.shape}")
                sin_comp = ensure_channels_first(sc[:, :, 0], force_layout=force_layout)
                cos_comp = ensure_channels_first(sc[:, :, 1], force_layout=force_layout)
                extras.extend([sin_comp, cos_comp])
                names.extend(["phase_sin", "phase_cos"])
            else:
                missing.update({"phase_sin", "phase_cos"})
                extras.extend([
                    np.zeros_like(amp, dtype=np.float32),
                    np.zeros_like(amp, dtype=np.float32),
                ])
                names.extend(["phase_sin", "phase_cos"])

    amp = amp.astype(np.float32)
    extras = [np.asarray(e, dtype=np.float32) for e in extras]
    return amp, extras, names, missing


def load_np_window(path, force_layout="TC"):
    return ensure_channels_first(np.load(path), force_layout=force_layout)


def compute_T_stats(paths, force_layout="TC", crop_len=None):
    T_list = []
    C_first = None
    for p in paths:
        X = np.load(p)
        if X.ndim != 2:
            continue
        if force_layout == "TC":
            C, T = X.shape[1], X.shape[0]
        else:
            C, T = X.shape
        if C_first is None:
            C_first = C
        T_list.append(T if crop_len is None else crop_len)
    if len(T_list) == 0:
        return C_first, None, None
    T_max = max(T_list) if crop_len is None else crop_len
    T_min = min(T_list) if crop_len is None else crop_len
    return C_first, T_min, T_max


def find_first_file(root, name):
    for r, _, files in os.walk(root):
        if name in files:
            return os.path.join(r, name)
    return None


def derive_mu_sigma_from_M(path_M, C_expected=None):
    M = np.load(path_M)  # [bins, C]
    if M.ndim != 2:
        raise ValueError(f"{path_M} is not 2D; got shape {M.shape}")
    bins, C_M = M.shape
    if C_expected is not None and C_M != C_expected:
        print(f"[WARN] avecsi_M C={C_M} != expected C={C_expected}; ignoring {path_M}")
        return None, None
    mu = M.mean(axis=0, keepdims=True).T.astype(np.float32)             # [C,1]
    sigma = (M.std(axis=0, keepdims=True).T + 1e-6).astype(np.float32)  # [C,1]
    return mu, sigma


def compute_avecsi_from_empty_windows(empty_paths, force_layout="TC"):
    if len(empty_paths) == 0:
        raise RuntimeError("No empty windows to compute AveCSI.")
    X0 = load_np_window(empty_paths[0], force_layout=force_layout)  # [C, T0]
    C = X0.shape[0]
    s1 = np.zeros((C,), dtype=np.float64)
    s2 = np.zeros((C,), dtype=np.float64)
    n  = np.zeros((C,), dtype=np.int64)
    s1 += X0.sum(axis=1); s2 += (X0**2).sum(axis=1); n += X0.shape[1]
    for p in empty_paths[1:]:
        x = load_np_window(p, force_layout=force_layout)
        if x.shape[0] != C:
            raise RuntimeError(f"Channel mismatch while computing AveCSI: {p} has C={x.shape[0]} != {C}")
        s1 += x.sum(axis=1); s2 += (x**2).sum(axis=1); n += x.shape[1]
    mu = (s1 / np.maximum(n,1))[:, None].astype(np.float32)
    var = (s2 / np.maximum(n,1)) - (mu[:,0].astype(np.float64))**2
    var = np.maximum(var, 1e-12)
    sigma = (np.sqrt(var)[:, None] + 1e-6).astype(np.float32)
    return mu, sigma



def load_or_compute_avecsi(base, train_root, force_layout="TC", class_names=("Empty","Stationary"), class_dir_map=None):
    empty_paths, _ = list_window_files(train_root, [class_names[0]], class_dir_map=class_dir_map)
    if len(empty_paths) == 0:
        raise RuntimeError("No 'empty' windows under training_set to compute AveCSI.")
    X0 = load_np_window(empty_paths[0], force_layout=force_layout)
    C_expected = X0.shape[0]

    if os.path.isfile(AVECSI_PATH):
        z = np.load(AVECSI_PATH)
        if z['mu'].shape[0] == C_expected:
            print(f"[AveCSI] Loaded {AVECSI_PATH} (C={C_expected})")
            return z['mu'].astype(np.float32), z['sigma'].astype(np.float32)

    matlab_fallback = os.path.join(base, MATLAB_AVE_NAME)
    if os.path.isfile(matlab_fallback):
        z = np.load(matlab_fallback)
        if z.shape[0] == C_expected:
            print(f"[AveCSI] Loaded MATLAB export {matlab_fallback} (C={C_expected})")
            mu = z.astype(np.float32)
            sigma = np.ones_like(mu, dtype=np.float32)
            np.savez_compressed(AVECSI_PATH, mu=mu, sigma=sigma)
            return mu, sigma

    all_paths, _ = list_window_files(train_root, class_names, class_dir_map=class_dir_map)
    sample = all_paths if len(all_paths) <= 4096 else all_paths[:4096]
    acc = [load_np_window(p, force_layout=force_layout) for p in sample]
    stacked = np.concatenate(acc, axis=1)
    mu = stacked.mean(axis=1).astype(np.float32)
    sigma = (stacked.std(axis=1) + 1e-6).astype(np.float32)
    np.savez_compressed(AVECSI_PATH, mu=mu, sigma=sigma)
    print(f"[AveCSI] Computed -> {AVECSI_PATH}")
    return mu, sigma


def center_crop(arr, length):
    if length is None or arr.shape[1] <= length:
        return arr
    start = max(0, (arr.shape[1] - length) // 2)
    return arr[:, start:start + length]


def pad_to_length(arr, length):
    if length is None:
        return arr
    out = np.zeros((arr.shape[0], length), dtype=arr.dtype)
    t = min(length, arr.shape[1])
    out[:, :t] = arr[:, :t]
    return out


def compute_anchor_phase_profile(empty_paths, force_layout="TC", crop_len=None):
    accum = []
    for p in empty_paths:
        phase_path = related_window_path(p, "Xphase_window_")
        if not os.path.isfile(phase_path):
            continue
        phase = ensure_channels_first(np.load(phase_path), force_layout=force_layout).astype(np.float32)
        if crop_len is not None:
            phase = center_crop(phase, crop_len)
        unwrapped = np.unwrap(phase, axis=1)
        accum.append(unwrapped.mean(axis=1))
    if not accum:
        raise RuntimeError("Unable to compute anchor phase profile; no valid phase windows located for the empty class.")
    anchor = np.mean(accum, axis=0).astype(np.float32)
    return anchor



def estimate_anchor_distance(phase_window, anchor_profile, wavelength):
    if phase_window is None or anchor_profile is None or wavelength is None:
        return np.nan
    unwrapped = np.unwrap(phase_window, axis=1)
    mean_phase = unwrapped.mean(axis=1)
    if anchor_profile.shape[0] != mean_phase.shape[0]:
        raise ValueError(
            f"Anchor profile channel count {anchor_profile.shape[0]} != phase window channels {mean_phase.shape[0]}"
        )
    delta = mean_phase - anchor_profile
    distance = float(np.mean(delta) * (wavelength / (2.0 * np.pi)))
    return distance


def estimate_anchor_distances(paths, anchor_profile, feature_mode="amp", force_layout="TC", crop_len=None, wavelength=None):
    paths = list(paths)
    if anchor_profile is None or not wants_raw_phase(feature_mode):
        if anchor_profile is None:
            print("[Anchor] No anchor profile available; skipping distance estimation.")
        else:
            print("[Anchor] FEATURE_MODE lacks raw phase; distance head disabled.")
        info = {"missing_phase": len(paths), "total": len(paths), "errors": 0}
        return np.full(len(paths), np.nan, dtype=np.float32), info
    if wavelength is None:
        wavelength = globals().get("WAVELENGTH_METERS")
    distances = []
    missing = 0
    errors = 0
    for p in paths:
        phase_path = related_window_path(p, "Xphase_window_")
        if not os.path.isfile(phase_path):
            distances.append(np.nan)
            missing += 1
            continue
        phase = ensure_channels_first(np.load(phase_path), force_layout=force_layout).astype(np.float32)
        if crop_len is not None:
            phase = center_crop(phase, crop_len)
            phase = pad_to_length(phase, crop_len)
        try:
            d = estimate_anchor_distance(phase, anchor_profile, wavelength) if wavelength is not None else np.nan
        except Exception as e:
            print(f"[WARN] Distance estimation failed for {os.path.basename(p)}: {e}")
            d = np.nan
            errors += 1
        distances.append(d)
    info = {"missing_phase": int(missing), "total": len(paths), "errors": int(errors)}
    return np.array(distances, dtype=np.float32), info


def compute_distance_scaler(distances):
    arr = np.asarray(distances, dtype=np.float32)
    if arr.size == 0:
        return {"mean": 0.0, "std": 1.0}
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return {"mean": 0.0, "std": 1.0}
    return {"mean": float(finite.mean()), "std": float(finite.std() + 1e-6)}

def build_array(paths, labels, mu=None, sigma=None, feature_mode="amp", extra_stats=None,
                force_layout="TC", crop_len=None, pad_to=None, anchor_profile=None, wavelength=None):
    extra_stats = extra_stats or {}
    arrays = []
    missing_counter = Counter()
    component_order = None
    distance_values = []
    distance_missing = 0

    if crop_len is None and pad_to is None:
        raise ValueError("pad_to must be provided when crop_len is None")

    for p in paths:
        amp, extras, names, missing = load_window_components(p, feature_mode, force_layout)
        if component_order is None:
            component_order = ["amp"] + names
        amp = amp.astype(np.float32)
        if mu is not None and sigma is not None:
            amp = (amp - mu) / (sigma + 1e-6)

        phase_for_distance = None
        if anchor_profile is not None and wavelength is not None and "phase" in names:
            idx = names.index("phase")
            phase_for_distance = extras[idx].astype(np.float32)

        parts = [amp]
        for name, arr in zip(names, extras):
            arr = arr.astype(np.float32)
            if name in missing:
                missing_counter[name] += 1
                arr = np.zeros_like(arr)
            elif name in extra_stats:
                mu_e, sigma_e = extra_stats[name]
                arr = (arr - mu_e) / (sigma_e + 1e-6)
            else:
                missing_counter[f"stats_missing::{name}"] += 1
            parts.append(arr)

        if crop_len is not None:
            parts = [center_crop(part, crop_len) for part in parts]
            if phase_for_distance is not None:
                phase_for_distance = center_crop(phase_for_distance, crop_len)

        target_pad = pad_to if pad_to is not None else crop_len
        parts = [pad_to_length(part, target_pad) for part in parts]

        x = np.concatenate(parts, axis=0)
        arrays.append(x.astype(np.float32))

        if anchor_profile is not None and wavelength is not None:
            if phase_for_distance is None:
                distance_values.append(np.nan)
                distance_missing += 1
            else:
                distance = estimate_anchor_distance(phase_for_distance, anchor_profile, wavelength)
                if np.isnan(distance):
                    distance_missing += 1
                    distance_values.append(np.nan)
                else:
                    distance_values.append(distance)
        else:
            distance_values.append(0.0)

    if not arrays:
        raise RuntimeError("No windows to build array.")

    X = np.stack(arrays, axis=0)
    y = np.array(labels, dtype=np.int32)
    distances = np.array(distance_values, dtype=np.float32)
    info = {
        "component_order": component_order if component_order is not None else ["amp"],
        "missing_features": dict(missing_counter),
        "available_feature_stats": sorted(extra_stats.keys()),
        "total_windows": len(arrays),
        "distance_missing": int(distance_missing),
    }
    return X, y, info, distances


## 3) Load lists, AveCSI, arrays, and anchor distance targets


In [30]:

train_paths, y_train = list_window_files(TRAIN_ROOT, CLASS_NAMES, class_dir_map=CLASS_FOLDERS)
val_paths,   y_val   = list_window_files(VAL_ROOT,   CLASS_NAMES, class_dir_map=CLASS_FOLDERS)
test_paths,  y_test  = list_window_files(TEST_ROOT,  CLASS_NAMES, class_dir_map=CLASS_FOLDERS)

if len(train_paths)==0 or len(val_paths)==0 or len(test_paths)==0:
    raise RuntimeError("One of the splits has 0 files. Check your folder structure and file names.")

C_train, T_train_min, T_train_max = compute_T_stats(train_paths, FORCE_LAYOUT, CROP_LEN)
C_val,   T_val_min,   T_val_max   = compute_T_stats(val_paths,   FORCE_LAYOUT, CROP_LEN)
C_test,  T_test_min,  T_test_max  = compute_T_stats(test_paths,  FORCE_LAYOUT, CROP_LEN)

if CROP_LEN is None:
    T_pad = max(t for t in [T_train_max, T_val_max, T_test_max] if t is not None)
else:
    T_pad = CROP_LEN

print(f"Detected C: train={C_train}, val={C_val}, test={C_test}")
print(f"Using common T = {T_pad}")

mu, sigma = load_or_compute_avecsi(BASE, TRAIN_ROOT, force_layout=FORCE_LAYOUT,
                                  class_names=CLASS_NAMES, class_dir_map=CLASS_FOLDERS)
assert mu.shape[0] == C_train, f"AveCSI C={mu.shape[0]} does not match data C={C_train}"
print("AveCSI shapes:", mu.shape, sigma.shape)

extra_stats = {}
missing_stats = []
if FEATURE_MODE != "amp":
    extra_stats = compute_extra_feature_stats(train_paths, FEATURE_MODE, force_layout=FORCE_LAYOUT, crop_len=CROP_LEN)
    expected = expected_feature_names(FEATURE_MODE)
    if extra_stats:
        print("[Phase] Computed stats for:", ", ".join(sorted(extra_stats.keys())))
    missing_stats = [name for name in expected if name not in extra_stats]
    if missing_stats:
        print(f"[WARN] Missing feature stats for {missing_stats}; zero-fill will be used for those channels.")

anchor_phase_profile = None
anchor_source_desc = None
if ENABLE_ANCHOR_DISTANCE:
    if wants_raw_phase(FEATURE_MODE):
        anchor_empty_class = ANCHOR_REFERENCE_EMPTY_CLASS or (CLASS_NAMES[0] if CLASS_NAMES else "Empty")
        try:
            anchor_empty_index = int(CLASS_NAMES.index(anchor_empty_class))
        except ValueError:
            anchor_empty_index = 0
            if CLASS_NAMES:
                print(f"[WARN] Anchor reference class '{ANCHOR_REFERENCE_EMPTY_CLASS}' not present in CLASS_NAMES; defaulting to '{CLASS_NAMES[0]}'.")
                anchor_empty_class = CLASS_NAMES[0]
            else:
                anchor_empty_class = "Empty"
        anchor_candidates = []
        if ANCHOR_REFERENCE_ROOT:
            if os.path.isdir(ANCHOR_REFERENCE_ROOT):
                anchor_dir_map = ANCHOR_CLASS_FOLDERS or CLASS_FOLDERS
                try:
                    anchor_candidates, _ = list_window_files(ANCHOR_REFERENCE_ROOT, [anchor_empty_class], class_dir_map=anchor_dir_map)
                except Exception as e:
                    print(f"[WARN] Unable to gather anchor windows from reference root '{ANCHOR_REFERENCE_ROOT}': {e}")
                    anchor_candidates = []
                if anchor_candidates:
                    anchor_source_desc = f"{ANCHOR_REFERENCE_ROOT} ({anchor_empty_class})"
                    print(f"[Anchor] Loaded {len(anchor_candidates)} reference windows for '{anchor_empty_class}' from {ANCHOR_REFERENCE_ROOT} to profile anchor '{ANCHOR_LOCATION_NAME}'.")
                else:
                    print(f"[WARN] No '{anchor_empty_class}' windows found under anchor reference root {ANCHOR_REFERENCE_ROOT}.")
            else:
                print(f"[WARN] Anchor reference root not found: {ANCHOR_REFERENCE_ROOT}")
        if not anchor_candidates:
            anchor_candidates = [p for p, y in zip(train_paths, y_train) if int(y) == anchor_empty_index]
            if anchor_candidates:
                anchor_source_desc = f"train split ({anchor_empty_class})"
                print(f"[Anchor] Falling back to {anchor_source_desc}; found {len(anchor_candidates)} candidate windows.")
            else:
                print("[WARN] Unable to locate any candidate windows for anchor profile computation.")
        if anchor_candidates:
            try:
                anchor_phase_profile = compute_anchor_phase_profile(anchor_candidates, force_layout=FORCE_LAYOUT, crop_len=CROP_LEN)
                print(f"[Anchor] Phase profile for '{ANCHOR_LOCATION_NAME}' computed from {anchor_source_desc}.")
            except Exception as e:
                print(f"[WARN] Unable to compute anchor phase profile: {e}")
                anchor_phase_profile = None
                anchor_source_desc = None
    else:
        print("[WARN] Anchor distance requested but FEATURE_MODE lacks raw phase; disabling distance head.")
        anchor_phase_profile = None
else:
    print("[Anchor] Distance head disabled via configuration.")

anchor_kwargs = {
    "anchor_profile": anchor_phase_profile if anchor_phase_profile is not None else None,
    "wavelength": WAVELENGTH_METERS if anchor_phase_profile is not None else None,
}

X_train, y_train, train_info, d_train_raw = build_array(train_paths, y_train, mu=mu, sigma=sigma,
                                           feature_mode=FEATURE_MODE, extra_stats=extra_stats,
                                           force_layout=FORCE_LAYOUT, crop_len=CROP_LEN, pad_to=T_pad,
                                           **anchor_kwargs)
X_val,   y_val,   val_info,   d_val_raw   = build_array(val_paths,   y_val,   mu=mu, sigma=sigma,
                                           feature_mode=FEATURE_MODE, extra_stats=extra_stats,
                                           force_layout=FORCE_LAYOUT, crop_len=CROP_LEN, pad_to=T_pad,
                                           **anchor_kwargs)
X_test,  y_test,  test_info,  d_test_raw  = build_array(test_paths,  y_test,  mu=mu, sigma=sigma,
                                           feature_mode=FEATURE_MODE, extra_stats=extra_stats,
                                           force_layout=FORCE_LAYOUT, crop_len=CROP_LEN, pad_to=T_pad,
                                           **anchor_kwargs)

if anchor_phase_profile is None:
    d_train_raw = d_val_raw = d_test_raw = None

DISTANCE_SCALER = None
if anchor_phase_profile is not None:
    DISTANCE_SCALER = compute_distance_scaler(d_train_raw)
    def _standardize(arr):
        arr = np.asarray(arr, dtype=np.float32)
        if STANDARDIZE_DISTANCE:
            arr = (arr - DISTANCE_SCALER["mean"]) / (DISTANCE_SCALER["std"] + 1e-6)
        return np.nan_to_num(arr, nan=0.0)
    d_train = _standardize(d_train_raw)
    d_val = _standardize(d_val_raw)
    d_test = _standardize(d_test_raw)
else:
    d_train = d_val = d_test = None

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape,   y_val.shape)
print("Test: ", X_test.shape,  y_test.shape)
print("Component order:", train_info.get("component_order"))

if train_info.get("missing_features"):
    print("[WARN] Missing features encountered in train:", train_info["missing_features"])
if val_info.get("missing_features"):
    print("[WARN] Missing features encountered in val:", val_info["missing_features"])
if test_info.get("missing_features"):
    print("[WARN] Missing features encountered in test:", test_info["missing_features"])
if missing_stats:
    print("[WARN] No global stats for:", missing_stats)

C = X_train.shape[1]
assert X_val.shape[1] == C and X_test.shape[1] == C, "Channel count mismatch across splits."

DISTANCE_METADATA = {
    "enabled": bool(anchor_phase_profile is not None),
    "scaler": DISTANCE_SCALER,
    "anchor_source": anchor_source_desc,
    "anchor_location_name": ANCHOR_LOCATION_NAME,
    "anchor_reference_root": ANCHOR_REFERENCE_ROOT,
    "anchor_reference_class": ANCHOR_REFERENCE_EMPTY_CLASS,
    "train_raw": d_train_raw.tolist() if anchor_phase_profile is not None else None,
    "val_raw": d_val_raw.tolist() if anchor_phase_profile is not None else None,
    "test_raw": d_test_raw.tolist() if anchor_phase_profile is not None else None,
}


path: Empty
path: Stationary
path: Empty
path: Stationary
path: Empty
path: Stationary
Detected C: train=106, val=106, test=106
Using common T = 4
path: Empty
[AveCSI] Computing from training_set/empty windows ...
[AveCSI] Saved to /home/tonyliao/WIFI_SENSING_LOCATION/PreCNN_Dataset/avecsi_empty.npz
AveCSI shapes: (106, 1) (106, 1)
Train: (22293, 106, 4) (22293,)
Val:   (7431, 106, 4) (7431,)
Test:  (7431, 106, 4) (7431,)


## 4) Model & train

In [31]:
def build_backbone_1d(in_channels: int, T: int, hidden: int = 128, wd: float = 1e-4, name="csi_backbone"):
    inp = layers.Input(shape=(in_channels, T), name="csi_in")
    x = layers.Permute((2,1), name="permute_T_C")(inp)
    x = layers.Conv1D(hidden, 7, padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(wd), name="conv1")(x)
    x = layers.BatchNormalization(name="bn1")(x)
    x = layers.MaxPooling1D(2, name="pool1")(x)
    x = layers.Conv1D(hidden, 5, padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(wd), name="conv2")(x)
    x = layers.BatchNormalization(name="bn2")(x)
    x = layers.MaxPooling1D(2, name="pool2")(x)
    x = layers.Dropout(0.2, name="drop2")(x)
    x = layers.Conv1D(hidden*2, 1, padding="same", activation="relu",
                      kernel_regularizer=regularizers.l2(wd), name="conv3")(x)
    x = layers.BatchNormalization(name="bn3")(x)
    h = layers.GlobalAveragePooling1D(name="gap")(x)
    h = layers.Dense(hidden, activation="relu", name="embed")(h)
    return models.Model(inp, h, name=name)

backbone = build_backbone_1d(in_channels=C, T=X_train.shape[2], hidden=128, wd=WEIGHT_DECAY)
logits   = layers.Dense(NUM_CLASSES, activation=None, name="logits")(backbone.outputs[0])

distance_head = None
if ENABLE_ANCHOR_DISTANCE and anchor_phase_profile is not None:
    distance_head = layers.Dense(1, activation=None, name="distance")(backbone.outputs[0])

if distance_head is not None:
    outputs = {"logits": logits, "distance": distance_head}
else:
    outputs = logits

model = models.Model(backbone.inputs[0], outputs, name="clf_windows_avecsi")

if distance_head is not None:
    losses = {
        "logits": keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        "distance": keras.losses.MeanSquaredError(),
    }
    metrics = {
        "logits": [keras.metrics.SparseCategoricalAccuracy(name="acc")],
        "distance": [keras.metrics.MeanAbsoluteError(name="mae")],
    }
    loss_weights = {"logits": 1.0, "distance": DISTANCE_LOSS_WEIGHT}
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss=losses,
        loss_weights=loss_weights,
        metrics=metrics,
    )
    train_targets = {"logits": y_train, "distance": d_train}
    val_targets = {"logits": y_val, "distance": d_val}
else:
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
    )
    train_targets = y_train
    val_targets = y_val

model.summary()

hist = model.fit(
    X_train, train_targets,
    validation_data=(X_val, val_targets),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

# Accuracy curves
if distance_head is not None:
    acc_key = "logits_acc"
    val_acc_key = "val_logits_acc"
else:
    acc_key = "acc"
    val_acc_key = "val_acc"

plt.figure(); plt.plot(hist.history.get(acc_key, [])); plt.plot(hist.history.get(val_acc_key, []))
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(['train','val']); plt.title('Accuracy')
plt.savefig(os.path.join(OUT_DIR, 'acc_curve.png')); plt.close()

# Loss curves
plt.figure(); plt.plot(hist.history.get('loss', [])); plt.plot(hist.history.get('val_loss', []))
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(['train','val']); plt.title('Loss')
plt.savefig(os.path.join(OUT_DIR, 'loss_curve.png')); plt.close()

if distance_head is not None and 'distance_mae' in hist.history:
    plt.figure(); plt.plot(hist.history['distance_mae']);
    if 'val_distance_mae' in hist.history:
        plt.plot(hist.history['val_distance_mae'])
    plt.xlabel('Epoch'); plt.ylabel('MAE (scaled units)');
    plt.legend(['train','val']); plt.title('Distance MAE')
    plt.savefig(os.path.join(OUT_DIR, 'distance_mae_curve.png'));
    plt.close()

model.save(os.path.join(OUT_DIR, "clf_windows_avecsi.h5"))
backbone.save(os.path.join(OUT_DIR, "encoder_windows_avecsi.h5"))
print("Saved models to", OUT_DIR)


I0000 00:00:1761733627.359071   53524 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22181 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "clf_windows_avecsi"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ csi_in (InputLayer)             │ (None, 106, 4)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ permute_T_C (Permute)           │ (None, 4, 106)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv1D)                  │ (None, 4, 128)         │        95,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1 (BatchNormalization)        │ (None, 4, 128)         │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling1D)            │ (None, 2, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv1D)                  │ (None, 2, 128)         │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2 (BatchNormalization)        │ (None, 2, 128)         │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling1D)            │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop2 (Dropout)                 │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv1D)                  │ (None, 1, 256)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn3 (BatchNormalization)        │ (None, 1, 256)         │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling1D)    │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embed (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ logits (Dense)                  │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 245,378 (958.51 KB)

 Trainable params: 244,354 (954.51 KB)

...

I0000 00:00:1761733630.255918   53701 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


164/175 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - acc: 0.9848 - loss: 0.0818

2025-10-29 10:27:11.232089: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_935', 4 bytes spill stores, 4 bytes spill loads



175/175 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - acc: 0.9856 - loss: 0.0793 - val_acc: 1.0000 - val_loss: 0.0363
Epoch 2/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - acc: 1.0000 - loss: 0.0349 - val_acc: 1.0000 - val_loss: 0.0330
Epoch 3/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - acc: 1.0000 - loss: 0.0324 - val_acc: 1.0000 - val_loss: 0.0304
Epoch 4/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - acc: 1.0000 - loss: 0.0298 - val_acc: 1.0000 - val_loss: 0.0278
Epoch 5/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - acc: 1.0000 - loss: 0.0271 - val_acc: 1.0000 - val_loss: 0.0251
Epoch 6/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - acc: 1.0000 - loss: 0.0245 - val_acc: 1.0000 - val_loss: 0.0225
Epoch 7/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - acc: 1.0000 - loss: 0.0219 - val_acc: 1.0000 - val_loss: 0.0201
Epoch 8/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - acc: 1.0000 - loss: 0.0196 - val_acc: 0.9997 - val_loss: 0.0195
Epoch 9/40
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - acc: 0.9998 

Saved models to runs_windows_supervised_avecsi


## 5) Evaluate classification & distance regression


In [32]:
preds = model.predict(X_test, verbose=0)
if isinstance(preds, dict):
    logits_pred = preds.get('logits')
    distance_pred = preds.get('distance')
elif isinstance(preds, (list, tuple)):
    if len(preds) == 2:
        logits_pred, distance_pred = preds
    else:
        logits_pred, distance_pred = preds[0], None
else:
    logits_pred, distance_pred = preds, None
pred = np.argmax(logits_pred, axis=1)
test_acc = (pred == y_test).mean()
print("Test Accuracy:", float(test_acc))

cm = confusion_matrix(y_test, pred)
print("Confusion Matrix:", cm)
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
disp.plot(); plt.title('Confusion Matrix (Test)')
plt.savefig(os.path.join(OUT_DIR, 'cm_test.png')); plt.close()

rep = classification_report(y_test, pred, target_names=CLASS_NAMES)
print("Classification Report:", rep)
with open(os.path.join(OUT_DIR, 'classification_report.txt'), 'w') as f:
    f.write(rep)

distance_mae_m = None
if distance_pred is not None and ENABLE_ANCHOR_DISTANCE and anchor_phase_profile is not None:
    pred_scaled = np.asarray(distance_pred.squeeze(), dtype=np.float32)
    target_scaled = np.asarray(d_test.squeeze(), dtype=np.float32)
    if DISTANCE_SCALER is not None and STANDARDIZE_DISTANCE:
        pred_m = pred_scaled * DISTANCE_SCALER["std"] + DISTANCE_SCALER["mean"]
        target_m = np.asarray(d_test_raw.squeeze(), dtype=np.float32) if d_test_raw is not None else target_scaled
    else:
        pred_m = pred_scaled
        target_m = target_scaled
    distance_mae_m = float(np.mean(np.abs(pred_m - target_m)))
    print(f"Distance MAE (meters): {distance_mae_m:.4f}")
    plt.figure()
    plt.scatter(target_m, pred_m, alpha=0.4)
    plt.xlabel('Target distance to anchor A (m)')
    plt.ylabel('Predicted distance to anchor A (m)')
    plt.title('Distance Regression (Test)')
    lims = [min(target_m.min(), pred_m.min()), max(target_m.max(), pred_m.max())]
    plt.plot(lims, lims, 'k--', linewidth=1)
    plt.savefig(os.path.join(OUT_DIR, 'distance_regression.png'))
    plt.close()

summary = {
    "base": BASE,
    "classes": CLASS_NAMES,
    "class_folders": CLASS_FOLDERS,
    "channels": int(C),
    "T_common": int(X_train.shape[2]),
    "pad_to": int(T_pad),
    "crop_len": None if CROP_LEN is None else int(CROP_LEN),
    "feature_mode": FEATURE_MODE,
    "feature_components": train_info.get("component_order", []),
    "feature_stats_present": sorted(extra_stats.keys()),
    "missing_feature_stats": missing_stats,
    "train_missing_features": train_info.get("missing_features", {}),
    "val_missing_features": val_info.get("missing_features", {}),
    "test_missing_features": test_info.get("missing_features", {}),
    "train_distance_missing": train_info.get("distance_missing", 0),
    "val_distance_missing": val_info.get("distance_missing", 0),
    "test_distance_missing": test_info.get("distance_missing", 0),
    "total_windows": {
        "train": int(train_info.get("total_windows", X_train.shape[0])),
        "val": int(val_info.get("total_windows", X_val.shape[0])),
        "test": int(test_info.get("total_windows", X_test.shape[0])),
    },
    "epochs": int(EPOCHS),
    "batch_size": int(BATCH_SIZE),
    "lr": float(LR),
    "weight_decay": float(WEIGHT_DECAY),
    "train_samples": int(X_train.shape[0]),
    "val_samples": int(X_val.shape[0]),
    "test_samples": int(X_test.shape[0]),
    "test_accuracy": test_acc,
    "avecsi_path": AVECSI_PATH,
    "distance_head": {
        "enabled": bool(distance_pred is not None and ENABLE_ANCHOR_DISTANCE and anchor_phase_profile is not None),
        "anchor_name": ANCHOR_LOCATION_NAME,
        "anchor_source": anchor_source_desc,
        "anchor_reference_root": ANCHOR_REFERENCE_ROOT,
        "anchor_reference_class": ANCHOR_REFERENCE_EMPTY_CLASS,
        "wavelength_m": float(WAVELENGTH_METERS),
        "loss_weight": float(DISTANCE_LOSS_WEIGHT if distance_pred is not None else 0.0),
        "scaler": DISTANCE_SCALER,
        "standardized": bool(STANDARDIZE_DISTANCE and anchor_phase_profile is not None),
        "test_mae_m": distance_mae_m,
        "metadata": DISTANCE_METADATA,
    },
}
if d_train_raw is not None:
    summary["distance_head"].update({
        "train_range_m": [float(d_train_raw.min()), float(d_train_raw.max())],
    })

with open(os.path.join(OUT_DIR, 'run_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print("Saved run summary and figures to:", OUT_DIR)


2025-10-29 10:27:32.336521: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_128', 4 bytes spill stores, 4 bytes spill loads



Test Accuracy: 1.0
Confusion Matrix:
 [[3738    0]
 [   0 3693]]

Classification Report:
               precision    recall  f1-score   support

       Empty       1.00      1.00      1.00      3738
  Stationary       1.00      1.00      1.00      3693

    accuracy                           1.00      7431
   macro avg       1.00      1.00      1.00      7431
weighted avg       1.00      1.00      1.00      7431

Saved run summary and figures to: runs_windows_supervised_avecsi
